## ***Introduction***

The Softmax activation function is used in neural networks when performing classification.

It transforms raw, unbounded neuron output values (logits) into a normalized probability distribution.

The output probabilities:

Are always non-negative.

Always sum to 1.

Provide context, unlike ReLU output such as [12, 99, 318], which is meaningless without normalization.

### ***Why We Need Softmax***
Problems with using ReLU/Linear outputs for classification

Unbounded: Values can be extremely large or small.

Not normalized: Outputs do not indicate relative likelihood of classes.

Exclusive: Each output neuron acts independently; no sense of distribution.

- Example:
[12, 99, 318]

This gives no idea which class is relatively more likely.

Softmax fixes this by:

Exponentiating all values.

Normalizing them.

Producing a probability distribution.


In [41]:
import numpy as np

### ***What Softmax Returns***

***Given raw outputs:***

[4.8, 1.21, 2.385]

***Softmax outputs:***

[0.8953, 0.0247, 0.0800]

***Properties:***

All values are between 0 and 1.

They sum to 1.

The index of the largest probability gives the predicted class.

The magnitude of the probability shows confidence.

### Mathematical Definition

For a vector of outputs \( \mathbf{z} = [z_1, z_2, \dots, z_n] \):

***
$$
\text{softmax}(z_j) = \frac{e^{z_j}}{\sum_{k=1}^{n} e^{z_k}}
$$
***

***Where:***

- \( e^{z_j} \) is the **numerator** (exponentiated output).
- \( \sum_{k=1}^{n} e^{z_k} \) is the **denominator** (sum of all exponentiated values).

## ***Step-by-Step Breakdown***

### ***1. Raw outputs (logits)***

layer_outputs = [4.8, 1.21, 2.385]

### ***2. Exponentiate each output***

Exponentiation ensures all values are positive.

Python version (manual)

Output:

[121.5104, 3.35348, 10.85906]

Meaning

Higher raw outputs → larger exponentiated values.

Negative values remain positive because exp never produces negatives.

Exponential is a monotonic function, so ordering is preserved.

In [42]:
E = 2.71828182846

exp_values = []

for output in layer_outputs:

    exp_values.append(E ** output)

### ***3. Normalize (convert into probabilities)***

Divide each exponentiated value by the sum of all exponentiated values.

norm_base = sum(exp_values)
norm_values = [v / norm_base for v in exp_values]


Result:

[0.89528, 0.024708, 0.080009]


These represent probabilities.

## ***Vectorized "NumPy" Implementation***

In [43]:
import numpy as np

layer_outputs = [4.8, 1.21, 2.385]

exp_values = np.exp(layer_outputs)
probabilities = exp_values / np.sum(exp_values)

print(probabilities)

[0.89528266 0.02470831 0.08000903]


Outputs:

[0.89528266 0.02470831 0.08000903]

## ***Why Exponentiation Helps***

### Without exponentiation:

Values could be negative or arbitrary:

[4.8, 1.21, -2.385]

***After normalization***:

[0.62, 0.15, -0.77]   # invalid negative probability

***With exponentiation***:

Always positive:

exp(-2.38) = 0.092

***Ensures***:

- Non-negative probabilities.

- Larger values become much larger.

- Small values shrink significantly.

This magnifies differences and ensures stable class ranking.

## ***Numerical Stability Problem***

Exponentials grow ***extremely fast***:

exp(1)    = 2.71
exp(10)   = 22026.46
exp(100)  = 2.688e43
exp(1000) = inf (overflow)

This causes "exploding values".

### ***Stability Fix: Subtract Max Logit***

***For inputs***:

[4.8, 1.21, 2.385]


***Subtract max (4.8)***:

[0, -3.59, -2.415]


***Exponentiate***:

[1.0, 0.0276, 0.0893]


***Normalize***:

[0.8956, 0.0247, 0.0800]


Same result, but entirely safe.

### ***Softmax for Batches***

Given a batch-shaped input inputs of shape (batch_size, neurons):

***Explanation:***

axis=1: sum each row separately.

keepdims=True: keeps column structure, enabling broadcasting.

In [44]:
inputs = [4.8, 1.21, 2.385]
#  exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
# probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)

### ***Understanding Axis***

In [45]:
layer_outputs = np.array([
    [4.8, 1.21, 2.385],
    [8.9, -1.81, 0.2],
    [1.41, 1.051, 0.026]
])


***axis=0 (column-wise sum)***
[15.11, 0.451, 2.611]

***axis=1 (row-wise sum)***
[8.395, 7.29, 2.487]

***axis=1, keepdims=True***
[[8.395],
 [7.29 ],
 [2.487]]


This enables proper broadcasting when dividing.

### ***Softmax Activation Class***

In [46]:
class Activation_Softmax:
    
    def forward(self, inputs):
        # subtract max for numerical stability
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        
        # normalize row-wise
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        
        self.output = probabilities

### ***Demonstration***

In [47]:
softmax = Activation_Softmax()
softmax.forward([[1, 2, 3]])
print(softmax.output)

[[0.09003057 0.24472847 0.66524096]]


***Result:***

[[0.09003 0.24473 0.66524]]


***Now shift the inputs:***

[-2, -1, 0]


***Same softmax result:***

[[0.09003 0.24473 0.66524]]


Softmax is shift-invariant.

### ***Why Scaling Inputs Matters***

If:

[1, 2, 3]

softmax = [0.09, 0.24, 0.66]

***But if scaled by 0.5:***

[0.5, 1, 1.5]

softmax = [0.18, 0.30, 0.50]

Even though both vectors point in the same direction, confidence changes.

This is why input scaling is crucial (covered in Chapter 22).

Using Softmax in a Network

In [48]:
# dense1 = Layer_Dense(2, 3)
# activation1 = Activation_ReLU()

# dense2 = Layer_Dense(3, 3)
# activation2 = Activation_Softmax()

# dense1.forward(X)
# activation1.forward(dense1.output)

# dense2.forward(activation1.output)
# activation2.forward(dense2.output)

# print(activation2.output[:5])


Output:

[[0.3333 0.3333 0.3333]
 ...
]

Since weights are random, the network initially predicts roughly uniform distribution over the classes.

### ***Prediction via Softmax (argmax)***

For each sample:

The neuron with the highest probability is the predicted class.

***Use:***

predictions = np.argmax(output, axis=1)


Confidence matters as much as the predicted class.

***Example:***

[0.45, 0.55] → prediction: class 1, confidence: 0.55

[0.32, 0.36, 0.32] → prediction: class 1, confidence: 0.36


Even though prediction is same, confidence is very different.

In [49]:
import numpy as np
import nnfs
from nnfs.datasets import spiral_data

# nnfs.init()

# Dense layer
class Layer_Dense:
    # Layer initialization
    def __init__(self, n_inputs, n_neurons):
        # Initialize weights and biases
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    
    # Forward pass
    def forward(self, inputs):
        # Calculate output values from inputs, weights and biases
        self.output = np.dot(inputs, self.weights) + self.biases


# ReLU activation
class Activation_ReLU:
    # Forward pass
    def forward(self, inputs):
        # Calculate output values from input
        self.output = np.maximum(0, inputs)


# Softmax activation
class Activation_Softmax:
    # Forward pass
    def forward(self, inputs):
        # Get unnormalized probabilities
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        # Normalize them for each sample
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        self.output = probabilities


# Create dataset
X, y = spiral_data(samples=100, classes=3)

# Create Dense layer with 2 input features and 3 output values
dense1 = Layer_Dense(2, 3)

# Create ReLU activation (to be used with Dense layer)
activation1 = Activation_ReLU()

# Create second Dense layer with 3 input features and 3 output values
dense2 = Layer_Dense(3, 3)

# Create Softmax activation (to be used with Dense layer)
activation2 = Activation_Softmax()

# Forward pass through first dense layer
dense1.forward(X)

# Forward pass through ReLU activation
activation1.forward(dense1.output)

# Forward pass through second dense layer
dense2.forward(activation1.output)

# Forward pass through Softmax activation
activation2.forward(dense2.output)

# Print first 5 samples of output (confidence scores)
print(activation2.output[:5])

[[0.33333334 0.33333334 0.33333334]
 [0.33333355 0.33333322 0.3333332 ]
 [0.33333382 0.33333313 0.3333331 ]
 [0.3333341  0.33333302 0.33333293]
 [0.33333433 0.3333329  0.33333278]]


                     Raw Outputs (Logits)
                   --------------------------------
                   |   4.8   |   1.21   |   2.385  |
                   --------------------------------
                                |
                                |  (1) Exponentiate
                                v
                -----------------------------------------
                | exp(4.8)=121.51 | exp(1.21)=3.35 | exp(2.385)=10.86 |
                -----------------------------------------
                                |
                                |  (2) Sum all exponentials
                                v
                        total = 121.51 + 3.35 + 10.86
                               = 135.72
                                |
                                |  (3) Normalize each value
                                v
            ----------------------------------------------------
            | 121.51/135.72 | 3.35/135.72 | 10.86/135.72 |
            ----------------------------------------------------
                    = 0.8953       0.0247        0.0800
                                |
                                |
                                v
                Final Softmax Output (Probability Distribution)
                ------------------------------------------------
                |   0.8953   |   0.0247   |   0.0800   |
                ------------------------------------------------
